# Competition Runbook — Plant Disease Classification

**3 submissions. Follow in order. Do not skip steps.**

| Step | What | Expected score |
|---|---|---|
| 1 | CLIP zero-shot | 60–75% |
| 2 | SigLIP 2 fine-tuned + TTA | 93–96% |
| 3 | SigLIP 2 + DINOv2 ensemble | 95–97% |

---

## Step 0 — Check environment

In [ ]:
import sys
print(f"Python: {sys.version}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram:.1f} GB")
else:
    print("WARNING: No GPU found. Training will be very slow.")

import transformers, datasets, PIL, sklearn, timm
print(f"transformers: {transformers.__version__}")
print(f"datasets: {datasets.__version__}")
print("All libraries OK.")

If any import fails, run this install cell:

In [ ]:
# Only run this if the cell above had import errors
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "transformers", "datasets", "huggingface_hub",
    "accelerate", "scikit-learn", "pillow", "timm"
], check=True)
print("Done. Re-run the check cell above.")

---
## Step 1 — Clone competition pack & set paths

In [ ]:
import os
from pathlib import Path

WORKSPACE   = Path("/workspace")
PACK_DIR    = WORKSPACE / "competition_pack"
DATA_TRAIN  = WORKSPACE / "data" / "train"
DATA_TEST   = WORKSPACE / "data" / "test"
SIGLIP_DIR  = WORKSPACE / "siglip2-model"
DINOV2_DIR  = WORKSPACE / "dinov2-model"
SUB_CLIP    = WORKSPACE / "submission_clip.csv"
SUB_SIGLIP  = WORKSPACE / "submission_siglip2.csv"
SUB_ENS     = WORKSPACE / "submission_ensemble.csv"
FINAL_SUB   = WORKSPACE / "submission.csv"

print(f"Workspace: {WORKSPACE}")
print(f"Pack dir:  {PACK_DIR}")

In [ ]:
import subprocess

if not PACK_DIR.exists():
    print("Cloning competition pack...")
    subprocess.run(
        ["git", "clone", "https://github.com/AleksCela/competition_pack.git", str(PACK_DIR)],
        cwd=str(WORKSPACE), check=True
    )
    print("Cloned.")
else:
    print("Pack already exists, pulling latest...")
    subprocess.run(["git", "pull"], cwd=str(PACK_DIR))

os.chdir(PACK_DIR / "advanced")
print(f"Working directory: {os.getcwd()}")

---
## Step 2 — Download data from HuggingFace

You need to be logged in to HuggingFace. Run the login cell if needed.

In [ ]:
# Login — paste your HF token when prompted
# Skip this if you already ran huggingface-cli login in a terminal
from huggingface_hub import login
login()

In [ ]:
from huggingface_hub import snapshot_download

if not DATA_TRAIN.exists() or not any(DATA_TRAIN.iterdir()):
    print("Downloading training data...")
    snapshot_download(
        "SmellsLikeAISpirit/plant-disease-train",
        repo_type="dataset",
        local_dir=str(DATA_TRAIN)
    )
    print("Train data ready.")
else:
    print(f"Train data already at {DATA_TRAIN}")

if not DATA_TEST.exists() or not any(DATA_TEST.iterdir()):
    print("Downloading test data...")
    snapshot_download(
        "SmellsLikeAISpirit/plant-disease-test",
        repo_type="dataset",
        local_dir=str(DATA_TEST)
    )
    print("Test data ready.")
else:
    print(f"Test data already at {DATA_TEST}")

In [ ]:
# Quick data sanity check
train_images = list(DATA_TRAIN.rglob("*.jpg")) + list(DATA_TRAIN.rglob("*.png"))
test_images  = list(DATA_TEST.rglob("*.jpg"))  + list(DATA_TEST.rglob("*.png"))
train_classes = [d.name for d in DATA_TRAIN.iterdir() if d.is_dir()]

print(f"Train images : {len(train_images)}")
print(f"Test images  : {len(test_images)}")
print(f"Train classes: {len(train_classes)}")
print(f"Classes: {sorted(train_classes)[:5]} ... (showing first 5)")

---
## Step 3 — SUBMISSION 1: CLIP zero-shot

No training. Gets you on the leaderboard in ~5 min. **Run this now.**

In [ ]:
# Run CLIP zero-shot inference
result = subprocess.run(
    [
        "python", "clip_zero_shot.py",
        "--test-dir",    str(DATA_TEST),
        "--output-csv",  str(SUB_CLIP),
        "--batch-size",  "32",
        "--model-name",  "openai/clip-vit-large-patch14",
    ],
    cwd=str(PACK_DIR / "advanced"),
    capture_output=False
)
print(f"Exit code: {result.returncode}")

In [ ]:
# Validate the CSV
subprocess.run(
    [
        "python", "final_submission_check.py",
        "--csv",      str(SUB_CLIP),
        "--test-dir", str(DATA_TEST),
    ],
    cwd=str(PACK_DIR)
)

In [ ]:
# Copy as current best submission
import shutil
shutil.copy(SUB_CLIP, FINAL_SUB)
print(f"submission.csv updated from CLIP zero-shot")

# Preview
import csv
with open(FINAL_SUB) as f:
    rows = list(csv.DictReader(f))
print(f"Total rows: {len(rows)}")
print("Sample predictions:")
for r in rows[:5]:
    print(f"  {r['id']:40s} -> {r['label']}")

**Download `/workspace/submission.csv` and submit to the leaderboard now.**

---
## Step 4 — Train SigLIP 2 (main model)

~50-60 min on A100 80GB. Start this immediately after submitting CLIP.

**Do not wait for it to finish before moving on — it runs in the background.**

In [ ]:
import torch

# Auto-select batch size based on available VRAM
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb >= 75:
        SIGLIP_BATCH = 64
    elif vram_gb >= 40:
        SIGLIP_BATCH = 32
    else:
        SIGLIP_BATCH = 16
else:
    SIGLIP_BATCH = 8

print(f"VRAM: {vram_gb:.1f} GB  →  batch size: {SIGLIP_BATCH}")

In [ ]:
# Train SigLIP 2 — this cell will run for ~50-60 min
# Output streams live to the cell below
subprocess.run(
    [
        "python", "train_vit_large.py",
        "--model",           "google/siglip2-so400m-patch14-384",
        "--data-dir",        str(DATA_TRAIN),
        "--max-images",      "0",
        "--batch-size",      str(SIGLIP_BATCH),
        "--eval-batch-size", str(SIGLIP_BATCH * 2),
        "--epochs",          "5",
        "--lr",              "1e-5",
        "--weighted-loss",
        "--no-phase1",
        "--output-dir",      str(SIGLIP_DIR),
    ],
    cwd=str(PACK_DIR / "advanced")
)

**If SigLIP 2 isn't on HuggingFace yet**, fall back to ViT-large by changing `--model` to `google/vit-large-patch16-224` and `--lr` to `2e-5`.

---
## Step 5 — SUBMISSION 2: SigLIP 2 with TTA + OOD protection

In [ ]:
siglip_best = SIGLIP_DIR / "best"
assert siglip_best.exists(), f"Model not found at {siglip_best} — did training finish?"
print(f"Model ready at {siglip_best}")

In [ ]:
# Predict with Test-Time Augmentation (6 views per image)
# --ood-threshold 0.5: anything <50% confident is classified as 'other'
subprocess.run(
    [
        "python", "predict_tta.py",
        "--model-dir",     str(siglip_best),
        "--test-dir",      str(DATA_TEST),
        "--output-csv",    str(SUB_SIGLIP),
        "--num-tta",       "6",
        "--ood-threshold", "0.5",
        "--batch-size",    "16",
    ],
    cwd=str(PACK_DIR / "advanced")
)

In [ ]:
# Validate
subprocess.run(
    [
        "python", "final_submission_check.py",
        "--csv",      str(SUB_SIGLIP),
        "--test-dir", str(DATA_TEST),
        "--model-dir", str(siglip_best),
    ],
    cwd=str(PACK_DIR)
)

In [ ]:
# Update submission
shutil.copy(SUB_SIGLIP, FINAL_SUB)
print("submission.csv updated from SigLIP 2 TTA")

# Show label distribution (should have a non-trivial 'other' count)
from collections import Counter
with open(FINAL_SUB) as f:
    labels = [r["label"] for r in csv.DictReader(f)]
top = Counter(labels).most_common(10)
print("Top predicted classes:")
for lbl, cnt in top:
    print(f"  {lbl:50s} {cnt}")

**Download `/workspace/submission.csv` and submit. This is likely your best score.**

---
## Step 6 — Train DINOv2 (second model for ensemble)

~55-65 min on A100 80GB. Start immediately after Submission 2.

In [ ]:
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb >= 75:
        DINO_BATCH = 32
    elif vram_gb >= 40:
        DINO_BATCH = 16
    else:
        DINO_BATCH = 8
else:
    DINO_BATCH = 4

print(f"DINOv2 batch size: {DINO_BATCH}")

In [ ]:
# Train DINOv2 — ~55-65 min
subprocess.run(
    [
        "python", "train_dinov2.py",
        "--model",      "facebook/dinov2-large",
        "--data-dir",   str(DATA_TRAIN),
        "--max-images", "0",
        "--batch-size", str(DINO_BATCH),
        "--epochs",     "5",
        "--lr",         "5e-6",
        "--grad-accum", "2",
        "--output-dir", str(DINOV2_DIR),
    ],
    cwd=str(PACK_DIR / "advanced")
)

---
## Step 7 — SUBMISSION 3: SigLIP 2 + DINOv2 Ensemble

In [ ]:
dinov2_best = DINOV2_DIR / "best"
assert siglip_best.exists(),  f"SigLIP 2 model not found: {siglip_best}"
assert dinov2_best.exists(), f"DINOv2 model not found: {dinov2_best}"
print("Both models ready for ensemble.")

In [ ]:
# Ensemble: average softmax probs from both models
# Weight SigLIP 2 higher (0.6) since it's the stronger model
subprocess.run(
    [
        "python", "ensemble_predict.py",
        "--model-dirs",    str(siglip_best), str(dinov2_best),
        "--model-types",   "hf", "dinov2",
        "--weights",       "0.6", "0.4",
        "--test-dir",      str(DATA_TEST),
        "--output-csv",    str(SUB_ENS),
        "--ood-threshold", "0.5",
        "--batch-size",    "8",
    ],
    cwd=str(PACK_DIR / "advanced")
)

In [ ]:
# Validate
subprocess.run(
    [
        "python", "final_submission_check.py",
        "--csv",      str(SUB_ENS),
        "--test-dir", str(DATA_TEST),
        "--model-dir", str(siglip_best),
    ],
    cwd=str(PACK_DIR)
)

In [ ]:
# Update submission
shutil.copy(SUB_ENS, FINAL_SUB)
print("submission.csv updated from ensemble")

with open(FINAL_SUB) as f:
    labels = [r["label"] for r in csv.DictReader(f)]
print(f"Total rows: {len(labels)}")
top = Counter(labels).most_common(10)
print("Top predicted classes:")
for lbl, cnt in top:
    print(f"  {lbl:50s} {cnt}")

**Download `/workspace/submission.csv` and submit.**

---
## Step 8 — Final decision

Go to the leaderboard. Your submissions are:

| # | File | Expected score |
|---|---|---|
| 1 | `submission_clip.csv` | 60–75% |
| 2 | `submission_siglip2.csv` | 93–96% |
| 3 | `submission_ensemble.csv` | 95–97% |

**Select your highest-scoring submission as final.**

If DINOv2 wasn't ready in time, keep Submission 2 as final. A partially-trained ensemble is worse than a fully-trained single model.

---
## Troubleshooting

In [ ]:
# Run this if you hit OOM during training
# Halve the batch size and double grad-accum to keep effective batch size the same
print("OOM fix: use --batch-size N//2 --grad-accum 2")
print()

# Check disk space
import shutil as _shutil
total, used, free = _shutil.disk_usage("/workspace")
print(f"Disk: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total ({free/1e9:.1f} GB free)")

# Check GPU memory right now
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated(0) / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    print(f"GPU mem: {alloc:.2f} GB allocated, {reserved:.2f} GB reserved")

In [ ]:
# Free GPU memory between training runs if needed
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory freed.")